This notebook, 02_database_integration_commodities.ipynb, facilitates the transition of processed commodity data from a flat CSV file into a structured PostgreSQL relational database. It focuses on data integrity, deduplication, and establishing a persistent storage layer for downstream machine learning and analytics.

## Technical Overview
The script automates the ETL (Extract, Transform, Load) process for commodity market data:Database: PostgreSQL.Libraries: psycopg2 (for database connectivity) and pandas (for data manipulation).
    Source Data: data/commodities_cleaned.csv.Core 
    Functionality: Implements an "Upsert" (Update or Insert) logic to ensure the database remains current without creating duplicate entries.

## Database Schema & Structure
The system creates a table named commodities with the following key attributes:CategoryFieldsIdentitiescommodity, category, unit, currencyPrice Dataprice, price_usd Performanceday_change, pct_change, weekly, monthly, ytd, yoyTemporaldate_full, scrape_date

    Constraints:Unique Constraint: A composite key is enforced on (commodity, unit, currency, date_full) to prevent redundant records for the same asset on the same day.
    Indexing: Optimizes query performance by indexing category, commodity, price_usd, and scrape_date.

## Data Processing Pipeline
The integration follows a three-step internal logic:

    Preprocessing & Cleaning: Strips whitespace from string columns.Standardizes date formats and handles missing values by co-referencing date_full and scrape_date.

    Deduplication: Identifies duplicate rows based on the unique keys. In the provided execution log, the script successfully deduplicated the dataset to  unique rows.

    Upsert Logic (ON CONFLICT):If a record already exists for a specific commodity on a specific date, the script updates the existing prices and change percentages rather than failing or creating a double entry.

In [3]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd
from datetime import datetime
import os

# ---------------------------
# PostgreSQL connection config
# ---------------------------
db_config = {
    'dbname': 'postgres',
    'user': 'postgres',
    'password': '22022000',
    'host': 'localhost',
    'port': '5432'
}

# ---------------------------
# Load your CSV
# ---------------------------
csv_file = "data/commodities_cleaned.csv"
if not os.path.exists(csv_file):
    raise FileNotFoundError(f"CSV not found: {csv_file}")

df = pd.read_csv(csv_file)
print(f"CSV loaded: {len(df)} rows, columns: {list(df.columns)}")

# ---------------------------
# Drop table if exists
# ---------------------------
def drop_table():
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS commodities CASCADE;")
    conn.commit()
    cur.close()
    conn.close()
    print("Dropped existing table (if any)")

# ---------------------------
# Create table schema
# ---------------------------
def create_table():
    create_query = """
    CREATE TABLE commodities (
        id SERIAL PRIMARY KEY,
        commodity VARCHAR(255) NOT NULL,
        category VARCHAR(200),
        unit VARCHAR(100),
        currency VARCHAR(10),
        price NUMERIC(12,4),
        price_usd NUMERIC(12,4),
        day_change NUMERIC(10,4),
        pct_change NUMERIC(10,4),
        weekly NUMERIC(10,4),
        monthly NUMERIC(10,4),
        ytd NUMERIC(10,4),
        yoy NUMERIC(10,4),
        date_full DATE,
        scrape_date DATE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE (commodity, unit, currency, date_full)
    );
    
    CREATE INDEX IF NOT EXISTS idx_commodities_category ON commodities(category);
    CREATE INDEX IF NOT EXISTS idx_commodities_commodity ON commodities(commodity);
    CREATE INDEX IF NOT EXISTS idx_commodities_price_usd ON commodities(price_usd);
    CREATE INDEX IF NOT EXISTS idx_commodities_scrape_date ON commodities(scrape_date);
    """
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    cur.execute(create_query)
    conn.commit()
    cur.close()
    conn.close()
    print("Table created with UNIQUE (commodity, unit, currency, date_full)")

# ---------------------------
# Insert / Upsert function
# ---------------------------
def insert_commodities(df):
    # Clean string columns
    for col in ["commodity", "category", "unit", "currency"]:
        df[col] = df[col].astype(str).str.strip()
    
    df["commodity"] = df["commodity"].str.slice(0, 255)
    df["category"] = df["category"].str.slice(0, 200)
    df["unit"] = df["unit"].str.slice(0, 100)
    df["currency"] = df["currency"].str.slice(0, 10)
    
    # Parse dates
    df["date_full"] = pd.to_datetime(df["date_full"], errors="coerce").dt.date
    df["scrape_date"] = pd.to_datetime(df["scrape_date"], errors="coerce").dt.date
    df["date_full"] = df["date_full"].fillna(df["scrape_date"])
    df = df.dropna(subset=["date_full", "scrape_date"])
    
    # Deduplicate by UNIQUE keys
    KEY_COLS = ["commodity", "unit", "currency", "date_full"]
    before = len(df)
    df = df.sort_values("scrape_date").drop_duplicates(KEY_COLS, keep="last")
    print(f"🧹 Deduplicated: {before} → {len(df)} rows")
    
    if df.empty:
        print("⚠ No rows to insert")
        return
    
    # Prepare records
    records = list(
        df[
            ["commodity", "price", "price_usd",
             "day_change", "pct_change",
             "weekly", "monthly", "ytd", "yoy",
             "category", "unit", "currency",
             "date_full", "scrape_date"]
        ].itertuples(index=False, name=None)
    )
    
    insert_query = """
    INSERT INTO commodities (
        commodity, price, price_usd,
        day_change, pct_change,
        weekly, monthly, ytd, yoy,
        category, unit, currency,
        date_full, scrape_date
    )
    VALUES %s
    ON CONFLICT (commodity, unit, currency, date_full)
    DO UPDATE SET
        price = EXCLUDED.price,
        price_usd = EXCLUDED.price_usd,
        day_change = EXCLUDED.day_change,
        pct_change = EXCLUDED.pct_change,
        weekly = EXCLUDED.weekly,
        monthly = EXCLUDED.monthly,
        ytd = EXCLUDED.ytd,
        yoy = EXCLUDED.yoy,
        category = EXCLUDED.category,
        scrape_date = EXCLUDED.scrape_date;
    """
    
    conn = psycopg2.connect(**db_config)
    cur = conn.cursor()
    try:
        execute_values(cur, insert_query, records)
        conn.commit()
        print(f"Inserted/updated {len(records)} rows successfully")
    except Exception as e:
        conn.rollback()
        print(f"Insert failed: {e}")
    finally:
        cur.close()
        conn.close()

# ---------------------------
# Run all steps
# ---------------------------
insert_commodities(df)


CSV loaded: 774 rows, columns: ['category', 'commodity', 'unit', 'price', 'day_change', 'pct_change', 'weekly', 'monthly', 'ytd', 'yoy', 'measure', 'date_raw', 'date_full', 'year', 'month', 'day', 'scrape_date', 'price_usd', 'id', 'date', 'currency']
🧹 Deduplicated: 773 → 773 rows
Inserted/updated 773 rows successfully



## Data Integration Summary
Automated ETL Pipeline: The script automates the extraction of commodity data, performing real-time cleaning and validation before loading it into a persistent storage environment.

Relational Schema Design: It defines a comprehensive table structure that captures essential market metrics, including pricing in USD, category classifications, and temporal performance indicators like pct_change and ytd.

Data Integrity & Validation: The pipeline includes a cleaning phase where string fields are stripped of whitespace and date formats are standardized to prevent database entry errors.

Composite Key Deduplication: To maintain a "single source of truth," the system uses a unique constraint on the combination of commodity, unit, currency, and date to identify and remove redundant records.

Upsert Implementation: The loading logic utilizes an ON CONFLICT clause, allowing the system to seamlessly update existing records with the latest market data or insert new entries as they arrive.

Optimized Querying: The architecture implements several database indices on high-traffic columns such as category and scrape_date to ensure fast retrieval for dashboards and machine learning models.

Successful Execution: During the most recent run, the system processed the dataset, successfully deduplicated the records, and updated the database with hundreds of validated rows.